# Anti-Goal Chess Benchmark @ NeurIPS 2026 — FULL RUN (paper data)

Paired win/lose small-model chess study (see README). Results land in `results/` and are zipped for download.
- Positions + exact oracles: committed (`data/positions/`), generated once by `scripts/generate_positions.py`.
- Engine + dataset tests gate every run: `scripts/test_engine.py`.
- Sweep: `scripts/run_suite.py` (models x tasks x {{win,lose}}).

## 1. Clone the repo

In [ ]:
import os, subprocess, sys
from pathlib import Path

WORK = Path("/kaggle/working")
REPO = WORK / "neuro-symbolic-pathfinding"
if not REPO.exists():
    url = "https://github.com/Vedang-P/neuro-symbolic-pathfinding.git"
    token = os.environ.get("GITHUB_TOKEN", "")
    if token:
        url = url.replace("https://", f"https://x-access-token:{token}@")
    subprocess.run(["git", "clone", "--quiet", url, str(REPO)], check=True)
os.chdir(REPO)
print("cwd:", Path.cwd())

## 2. Submodule + dependencies

In [ ]:
subprocess.run(["git", "submodule", "update", "--init", "--depth", "1"], check=True, capture_output=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-r", "requirements.txt"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "python-chess"], check=True)
subprocess.run([sys.executable, "-m", "pip", "uninstall", "--quiet", "-y", "wandb"], check=True)
import torch
print("torch", torch.__version__, "cuda", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

## 3. Stage runner (never raises; the verdict cell checks results)

In [ ]:
import json, time, shutil
from pathlib import Path

STAGE_LOG = Path("results/stage_log.json")
def run_stage(name, args, timeout_min):
    Path("results").mkdir(parents=True, exist_ok=True)
    rec = {"stage": name, "status": "running", "elapsed_min": None}
    t0 = time.time()
    try:
        res = subprocess.run(args, timeout=timeout_min * 60)
        rec["status"] = "ok" if res.returncode == 0 else "failed"
        rec["returncode"] = res.returncode
    except subprocess.TimeoutExpired:
        rec["status"] = "timeout"
    except Exception as e:
        rec["status"] = "error"
        rec["error"] = str(e)[:200]
    rec["elapsed_min"] = round((time.time() - t0) / 60, 1)
    entries = json.loads(STAGE_LOG.read_text()) if STAGE_LOG.exists() else []
    entries.append(rec)
    STAGE_LOG.write_text(json.dumps(entries, indent=1))
    print(f"stage {name}: {rec['status']} ({rec['elapsed_min']}min)", flush=True)
    return rec["status"]

## 4. Gate: engine + dataset tests

In [ ]:
status = run_stage("engine_tests", [sys.executable, "scripts/test_engine.py", "--quick"], 60)
if status != "ok":
    raise RuntimeError("engine tests failed -- see output above")

## 5. Data validation

In [ ]:
import json
for name in ["sm-3x3-win", "sm-3x3-draw", "sm-5x5-win", "sm-5x5-draw", "mate1-8x8", "mob-8x8"]:
    recs = json.loads(Path(f"data/positions/{name}.json").read_text())
    assert len(recs) >= 40, f"{name}: expected >=40 positions, got {len(recs)}"
    assert all("win_moves" in r and "lose_moves" in r for r in recs)
print("committed position data OK (all 6 task sets, oracle fields present)")

## 6. The chess sweep (models x tasks, paired win/lose)

In [ ]:
status = run_stage(
    "chess_sweep",
    [sys.executable, "scripts/run_suite.py", "--output_dir", "results/chess"],
    720,
)
print("sweep:", status)

## 7. Results table

In [ ]:
import pandas as pd
csv_path = Path("results/chess/comparison_table.csv")
if csv_path.exists():
    df = pd.read_csv(csv_path)
    display(df)
    print("rows:", len(df))
else:
    print("no comparison table -- sweep did not complete")

## 8. Zip results

In [ ]:
shutil.make_archive("/kaggle/working/results", "zip", root_dir=Path("results").resolve())
print("zipped results.zip")

## Notes
- **Resume after a died session:** re-run the notebook with a trimmed sweep, e.g. `run_suite.py --models <remaining> --tasks <remaining> --output_dir results/chess`; per-run JSONs under `results/chess/*.summary.json` are the source of truth; the CSV is rebuilt at the end.
- **Gemma models** need the `HF_TOKEN` Kaggle secret (gated access).
- **Timeouts:** full-mode sweep is capped at 12h; typical T4 estimate ~1-2 min/position-cell, well under a single Kaggle session.